### Importing packages

In [55]:
import pandas as pd              # type: ignore     
import numpy as np               # type: ignore

### Importing dataset

In [56]:
test_data_X = pd.read_csv("../obj-to-map/results-merged/test-data-zambia-input-ob-t0.csv")
test_data_y_lt0 = pd.read_csv("../obj-to-map/results-merged/test-data-zambia-map-t0.csv", header=None)
test_data_y_lt1 = pd.read_csv("../obj-to-map/results-merged/test-data-zambia-map-t1.csv", header=None)

In [57]:
train_data_X = pd.read_csv("../obj-to-map/results-merged/train-data-zambia-input-ob-t0.csv")
train_data_y_lt0 = pd.read_csv("../obj-to-map/results-merged/train-data-zambia-map-t0.csv", header=None)
train_data_y_lt1 = pd.read_csv("../obj-to-map/results-merged/train-data-zambia-map-t1.csv", header=None)

### Choosing lead time

In [58]:
def downsample_grid(grid, factor):
    # Ensure that the grid dimensions are divisible by the factor
    assert grid.shape[0] % factor == 0
    assert grid.shape[1] % factor == 0

    # Reshape the grid to factor blocks and then take the mean
    grid_downsampled = grid.reshape(grid.shape[0] // factor, factor,
                                    grid.shape[1] // factor, factor).mean(axis=(1, 3))
    return grid_downsampled

In [59]:
lead_time = 1

### Choosing data split

In [60]:
dataset = "test"
if dataset == "train":
    data = train_data_X
    target = train_data_y_lt1
    target_t0 = train_data_y_lt0
else:
    data = test_data_X
    target = test_data_y_lt1
    target_t0 = test_data_y_lt0

In [61]:
original_data = data.copy()

### Finding exact rows after given time (hh, mm)

In [62]:
data

,year,month,day,hour,minute,lat,lon,wp,size,ds
0,2020,1,1,0,0,-14.3285,32.3142,170,3753,117.61
1,2020,1,1,0,15,-14.6443,32.0499,98,16542,108.00
2,2020,1,1,0,30,-14.3228,32.0958,150,15858,111.65
3,2020,1,1,0,45,-14.3467,31.8838,131,14490,105.57
4,2020,1,1,1,0,-14.3438,31.7752,112,12105,102.59
...,...,...,...,...,...,...,...,...,...,...
46558,2024,1,25,18,15,-10.4485,24.1929,132,135,177.00
46559,2024,1,25,21,45,-10.4403,25.5046,133,198,156.59
46560,2024,1,25,22,30,-10.4612,25.0240,207,117,162.61
46561,2024,1,25,22,45,-10.4333,25.0533,194,315,162.98


In [63]:
data['datetime'] = pd.to_datetime(data[['year', 'month', 'day', 'hour', 'minute']])

In [64]:
def find_exact_row_after_given_hours(row, hours, minutes, df):
    target_time = row['datetime'] + pd.Timedelta(hours=hours, minutes=minutes)
    corresponding_row = df[df['datetime'] == target_time]
    if not corresponding_row.empty:
        return corresponding_row.index[0]  # Return the index of the corresponding row
    else:
        return None

In [65]:
data['row_index_X0_30'] = data.apply(find_exact_row_after_given_hours, args=(0, 30, data), axis=1)
data['row_index_X0_60'] = data.apply(find_exact_row_after_given_hours, args=(1, 0, data), axis=1)
data['row_index_X0_90'] = data.apply(find_exact_row_after_given_hours, args=(1, 30, data), axis=1)
data['row_index_X0_120'] = data.apply(find_exact_row_after_given_hours, args=(2, 0, data), axis=1)

One hour lead time

This is relative to 120 as it's the nowcast origin

+1 as $t_0$ is at 120 minutes

### Find the target index for the chosen lead time for each row

We need $t_0$, $t_1$ and $t_{lt}$

This it to get the corresponding target value after one hour

In [66]:
data['row_index_Cb'] = data.apply(find_exact_row_after_given_hours, args=(lead_time + 1 , 0, data), axis=1)      # since the target is at t0+1 h

In [67]:
columns_to_keep = original_data.keys().to_list()

In [68]:
# Function to combine current row with rows based on indices, retaining original column order
def combine_current_and_rows(row, data):
    # Extract the row indices from the current row
    indices = [row['row_index_X0_30'], row['row_index_X0_60'], row['row_index_X0_90'], row['row_index_X0_120']]
    
    # Fetch the current row (filter only relevant columns)
    current_row = row[columns_to_keep].copy()
    
    # List to store the rows
    rows_to_combine = [current_row]
    
    # Fetch rows corresponding to the indices, rename columns with suffix to avoid duplicates
    for i, idx in enumerate(indices):
        if pd.notna(idx):
            # Fetch the row, keep only relevant columns, and rename them with suffix
            fetched_row = data.loc[idx, columns_to_keep].rename(lambda col: f"{col}_{(i+1)*30}")
            rows_to_combine.append(fetched_row)
        else:
            # If the index is NaN, create an empty Series with the same columns as the current row
            empty_row = pd.Series(index=[f"{col}_{(i+1)*30}" for col in columns_to_keep])
            rows_to_combine.append(empty_row)
    
    # Concatenate the current row with the fetched rows side by side (maintain column order)
    combined_row = pd.concat(rows_to_combine, axis=0)
    
    return combined_row
# Apply the function row by row to get combined data
combined_data = data.apply(combine_current_and_rows, args=(data,), axis=1)

# Convert the combined series into a DataFrame while keeping the original index
combined_df = pd.DataFrame(combined_data, index=data.index)

### Taking target values

In [69]:
combined_df = pd.DataFrame(combined_data, index=data.index)

In [70]:
combined_df = combined_df.dropna()

In [71]:
target_index = data['row_index_Cb'][combined_df.index].dropna()
target = target.loc[target_index]
target_t0 = target_t0.loc[target_index]
combined_df = combined_df.loc[target_index.index]

In [72]:
target = target.to_numpy()
target_t0 = target_t0.to_numpy()

In [73]:
target.shape

(38128, 598)

The new image shape is 23 x 26

23 being the height

In [74]:
598/26

23.0

In [75]:
target = target.reshape(len(target), 23, 26)
target_t0 = target_t0.reshape(len(target_t0), 23, 26)

In [76]:
combined_df.to_csv(f"../obj-to-map/final/{dataset}-ob-to-map-input-t0-lt{lead_time}.csv", index=False)

In [77]:
np.save(f"../obj-to-map/final/{dataset}-ob-to-map-target-lt{lead_time}.npy", target)
np.save(f"../obj-to-map/final/{dataset}-ob-to-map-target-lt0-lt{lead_time}.npy", target_t0)

### Writing data to disk